# HW 3.1 — Graph Search: BFS vs. UCS
### AI 680 / CS 666 · Kyle Allen Sherman

---

This notebook walks through the logic of each section of `sherman_hw_3_1.py`. The goal is not just to *run* the code but to understand **why** each design decision was made and how it connects to the theory from Russell & Norvig (Ch. 3).

**What the code does:**  
It defines a weighted, directed graph and finds a path from node `A` to node `G` using two different search strategies:
- **BFS** — finds the path with the *fewest edges* (ignores weights)
- **UCS / Dijkstra** — finds the path with the *lowest total cost* (respects weights)

Then it plots both results side-by-side for comparison.

---
## Section 1 — Imports

```python
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import heapq
```

| Import | Why it's here |
|---|---|
| `matplotlib.pyplot` | Draws the graph visualisation |
| `matplotlib.patches` | Creates the colored legend boxes |
| `collections.deque` | Provides the **FIFO queue** BFS requires. `deque.popleft()` is O(1); a plain list's `pop(0)` is O(n) |
| `heapq` | Provides the **min-heap priority queue** UCS requires — always pops the lowest-cost node next |

> **Key concept:** The data structure *is* the strategy. A FIFO queue → BFS. A min-heap → UCS. This is the same insight from the LIFO/FIFO visualization.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import heapq

---
## Section 2 — Graph Definition

### 2a. Edge list

The graph is first described as a flat list of **tuples**: `(source, destination, weight)`.  
This is a natural, human-readable format — you can read each line as a sentence: *"It costs 5 to travel from A to D."*

The graph is **directed** (arrows go one way) and **weighted** (each edge has a cost).

In [ ]:
edges = [
    ('A', 'D', 5),
    ('A', 'N', 3),
    ('D', 'C', 4),
    ('D', 'B', 4),
    ('N', 'B', 6),
    ('N', 'S', 2),
    ('B', 'C', 2),
    ('B', 'F', 3),
    ('B', 'E', 5),
    ('C', 'E', 5),
    ('E', 'G', 3),
    ('F', 'G', 4),
    ('F', 'H', 5),
    ('H', 'G', 6),
    ('S', 'H', 5),
]

### 2b. Building the Adjacency List

The edge list is convenient to write but slow to search. Search algorithms need to answer the question **"what are the neighbors of node X?"** repeatedly and quickly. An **adjacency list** (a dictionary) makes that O(1).

```python
graph = {}
for u, v, w in edges:
    graph.setdefault(u, []).append((v, w))
    graph.setdefault(v, [])   # ensure every node appears, even with no outgoing edges
```

**Line by line:**
- `graph.setdefault(u, [])` — if key `u` doesn't exist yet, create it with an empty list; then `.append((v, w))` adds the neighbor and weight.
- `graph.setdefault(v, [])` — ensures destination nodes (like `G`, which has no outgoing edges) still appear as keys. Without this, `graph.get('G', [])` would return `[]` correctly, but the node would be invisible to any code that iterates over `graph.keys()`.

**Result:** `graph['A']` → `[('D', 5), ('N', 3)]`  
**Result:** `graph['G']` → `[]` (goal node, no outgoing edges)

In [ ]:
graph = {}
for u, v, w in edges:
    graph.setdefault(u, []).append((v, w))
    graph.setdefault(v, [])

# Inspect the result
for node in sorted(graph):
    print(f"  {node}: {graph[node]}")

### 2c. Start, Goal, and Node Positions

`start` and `goal` are straightforward. The `pos` dictionary assigns each node an (x, y) coordinate — this is **only used for drawing** and has no effect on the search logic.

In [ ]:
start = 'A'
goal  = 'G'

pos = {
    'A': (0, 2),
    'N': (2, 1),
    'D': (2, 3),
    'S': (4, 0),
    'B': (4, 2),
    'C': (4, 4),
    'F': (6, 1),
    'E': (6, 3),
    'H': (8, 0),
    'G': (8, 2),
}

---
## Section 3 — BFS (Breadth-First Search)

### What BFS optimizes for
BFS explores nodes **level by level** (closest first). In an unweighted graph this guarantees the **shortest path by edge count**. In a weighted graph like this one, it finds the fewest-hop path, which may not be the cheapest.

### The frontier: `deque`
The FIFO queue is the heart of BFS. Nodes added early get processed early — the algorithm can never "jump ahead" to a deeper node before finishing shallower ones.

```python
queue = deque([(start, [start])])
```
Each item in the queue is a **tuple of (current_node, path_so_far)**. Carrying the path along avoids needing a separate "came from" dictionary to reconstruct it later.

### The `explored` set
```python
explored = set()
```
Prevents revisiting nodes. Without it, BFS would loop in cycles. A `set` is used (not a list) because membership testing — `if current not in explored` — is O(1) for sets and O(n) for lists.

### The loop logic
```python
current, path = queue.popleft()   # FIFO: oldest node first
if current == goal: return path   # goal test on pop, not on push
if current not in explored:
    explored.add(current)
    for neighbor, _ in graph.get(current, []):   # _ discards the weight
        if neighbor not in explored:
            queue.append((neighbor, path + [neighbor]))
```

Note `_` — BFS doesn't care about weights, so the weight is intentionally discarded with the underscore convention. Every edge is treated as cost 1.

In [ ]:
def bfs(graph, start, goal):
    """BFS: finds the path with the fewest edges (ignores edge weights)."""
    queue    = deque([(start, [start])])
    explored = set()

    while queue:
        current, path = queue.popleft()       # FIFO — oldest node out first
        if current == goal:
            return path
        if current not in explored:
            explored.add(current)
            for neighbor, _ in graph.get(current, []):   # _ = weight, ignored
                if neighbor not in explored:
                    queue.append((neighbor, path + [neighbor]))
    return None

# Quick test
bfs_result = bfs(graph, start, goal)
print("BFS path:", ' → '.join(bfs_result))
print("Edge count:", len(bfs_result) - 1)

---
## Section 4 — UCS / Dijkstra (Uniform-Cost Search)

### What UCS optimizes for
UCS always expands the **lowest-cost node on the frontier** next. This guarantees finding the optimal (cheapest) path in a graph with non-negative edge weights. It is equivalent to Dijkstra's algorithm.

### The frontier: `heapq` (min-heap)
```python
heap = [(0, start, [start])]
```
Each entry is `(cumulative_cost, node, path)`. Python's `heapq` always pops the **smallest** tuple — since the first element is cost, the cheapest node always comes out first.

### Why `explored` works differently here
```python
explored = {}   # node -> best cost seen
```
Unlike BFS, UCS uses a **dictionary** (not a set) because a node can appear on the heap multiple times with different costs. When a node is popped, we check:
```python
if current in explored and explored[current] <= cost:
    continue   # already found a cheaper path to this node — skip
```
This is the "lazy deletion" pattern — stale heap entries are left in place and discarded when popped.

### The loop logic
```python
cost, current, path = heapq.heappop(heap)    # always the cheapest node
if current == goal: return path, cost
explored[current] = cost
for neighbor, weight in graph.get(current, []):
    new_cost = cost + weight
    if neighbor not in explored or explored[neighbor] > new_cost:
        heapq.heappush(heap, (new_cost, neighbor, path + [neighbor]))
```

**Key difference from BFS:** every neighbor's cumulative cost is computed and stored. The heap re-sorts automatically so the cheapest path is always next.

In [ ]:
def ucs(graph, start, goal):
    """UCS: finds the path with the lowest total cost (respects edge weights)."""
    heap     = [(0, start, [start])]   # (cost, node, path)
    explored = {}                      # node -> best cost confirmed so far

    while heap:
        cost, current, path = heapq.heappop(heap)   # min-heap: cheapest node first
        if current == goal:
            return path, cost
        if current in explored and explored[current] <= cost:
            continue                  # stale entry — a cheaper path was already processed
        explored[current] = cost
        for neighbor, weight in graph.get(current, []):
            new_cost = cost + weight
            if neighbor not in explored or explored[neighbor] > new_cost:
                heapq.heappush(heap, (new_cost, neighbor, path + [neighbor]))
    return None, float('inf')

# Quick test
ucs_path, ucs_cost = ucs(graph, start, goal)
print("UCS path:", ' → '.join(ucs_path))
print("Total cost:", ucs_cost)

---
## Section 5 — Comparing the Two Results

Before plotting, the helper `compute_path_cost` calculates the actual edge-weight total for any given path. This is used to show BFS's cost even though BFS didn't use it during search.

In [ ]:
def compute_path_cost(path, graph):
    """Sum the edge weights along a path. Works for any path regardless of how it was found."""
    total    = 0
    edge_map = {(u, v): w for u, v, w in edges}   # fast lookup: (u,v) -> weight
    for i in range(len(path) - 1):
        total += edge_map[(path[i], path[i+1])]
    return total

bfs_path = bfs(graph, start, goal)
ucs_path, ucs_cost = ucs(graph, start, goal)

print("=" * 50)
print(f"  Start: {start}   Goal: {goal}")
print("=" * 50)

print("\n--- BFS (fewest edges) ---")
print(f"  Path  : {' → '.join(bfs_path)}")
print(f"  Edges : {len(bfs_path) - 1}")
print(f"  Cost  : {compute_path_cost(bfs_path, graph)}")

print("\n--- UCS / Dijkstra (lowest cost) ---")
print(f"  Path  : {' → '.join(ucs_path)}")
print(f"  Edges : {len(ucs_path) - 1}")
print(f"  Cost  : {ucs_cost}")

---
## Section 6 — Visualisation

The `plot_graph` function creates a side-by-side figure. A few things worth understanding:

- **`path_edges = set(zip(path, path[1:]))`** — converts `['A','B','C']` into `{('A','B'), ('B','C')}`. This makes it O(1) to check whether any given edge is on the path.
- **`ax.annotate(..., arrowprops=...)`** — draws directed arrows. `shrinkA` and `shrinkB` pull the arrowhead back from the node centers so the circle outline stays visible.
- **`plt.Circle`** — draws each node as a filled circle; `zorder=3/4` ensures nodes render on top of edges.
- Colors: green = start, blue = goal, salmon = intermediate path node, red arrow = path edge, light gray = non-path edge.

In [ ]:
def plot_graph(bfs_path, ucs_path, ucs_cost):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    titles = [
        f"BFS  (fewest edges)\nPath: {' → '.join(bfs_path)}   Cost: {compute_path_cost(bfs_path, graph)}",
        f"UCS / Dijkstra  (lowest cost)\nPath: {' → '.join(ucs_path)}   Cost: {ucs_cost}",
    ]
    paths = [bfs_path, ucs_path]

    for ax, path, title in zip(axes, paths, titles):
        # Set of (u,v) tuples on the solution path — O(1) membership test
        path_edges = set(zip(path, path[1:]))

        # Draw all edges
        for u, v, w in edges:
            x1, y1 = pos[u]
            x2, y2 = pos[v]
            color     = 'red'      if (u, v) in path_edges else 'lightgray'
            linewidth = 2.5        if (u, v) in path_edges else 1.0
            ax.annotate(
                "", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(
                    arrowstyle="-|>",
                    color=color,
                    lw=linewidth,
                    mutation_scale=18,
                    shrinkA=14, shrinkB=14,   # pull back from node centers
                )
            )
            # Weight label at edge midpoint
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            ax.text(mx, my, str(w), fontsize=8, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.7))

        # Draw nodes
        for node, (x, y) in pos.items():
            if node == start:
                color = 'green'
            elif node == goal:
                color = 'royalblue'
            elif node in path:
                color = 'salmon'
            else:
                color = 'whitesmoke'
            circle = plt.Circle((x, y), 0.55, color=color, ec='black', linewidth=1.5, zorder=3)
            ax.add_patch(circle)
            ax.text(x, y, node, ha='center', va='center', fontsize=12,
                    fontweight='bold', zorder=4)

        ax.set_xlim(-1, 10)
        ax.set_ylim(-1, 5.5)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(title, fontsize=11, pad=12)

        legend_handles = [
            mpatches.Patch(color='green',     label='Start (A)'),
            mpatches.Patch(color='royalblue', label='Goal (G)'),
            mpatches.Patch(color='salmon',    label='Path node'),
            mpatches.Patch(color='red',       label='Path edge'),
        ]
        ax.legend(handles=legend_handles, loc='lower right', fontsize=8)

    plt.suptitle("Graph Search: A → G", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_graph(bfs_path, ucs_path, ucs_cost)

---
## Summary: BFS vs. UCS at a Glance

| Property | BFS | UCS |
|---|---|---|
| **Frontier structure** | FIFO `deque` | Min-heap (`heapq`) |
| **Expansion order** | Fewest edges first | Lowest cumulative cost first |
| **Optimality guarantee** | Shortest path by *edge count* | Shortest path by *total weight* |
| **Handles weights?** | No — ignores them | Yes |
| **Equivalent to** | BFS (R&N Ch. 3) | Dijkstra's algorithm |
| **When paths differ** | BFS path may be cheaper on hops but costlier in weight | UCS path is always cost-optimal |

### Connection to your OULAD project
The same trade-off appears in temporal prediction modeling. BFS is like using the **earliest available prediction** (fewest time steps); UCS is like finding the **optimal intervention point** that minimizes total cost (false positive rate × resource expenditure). The "graph" in your project is the course timeline, and the edge weights are the costs of acting (or failing to act) at each day.